In [1]:
# ---------------------------------------------------------
# COPY AND RUN THIS CELL IN YOUR GOOGLE COLAB NOTEBOOK
# ---------------------------------------------------------
# 1. Install dependencies
!pip install flask pyngrok accelerate bitsandbytes
# 2. Authenticate ngrok (You need a free account)
# REPLACE 'YOUR_AUTHTOKEN' WITH YOUR ACTUAL TOKEN from https://dashboard.ngrok.com/get-started/your-authtoken
get_ipython().system_raw('ngrok config add-authtoken 39eTQZbRRwQbx8MPXzI6O8EKG5a_yqGWmRNvosuxEYHvvns7')
# 3. Load Model (Assuming you already loaded it in previous cells)
# If not, uncomment below:
# from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
# import torch
# model_id = "meta-llama/Meta-Llama-3-8B-Instruct"
# pipe = pipeline("text-generation", model=model_id, model_kwargs={"torch_dtype": torch.bfloat16}, device_map="auto")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 16.6 MB/s eta 0:00:00


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TextStreamer

# --- CHANGE IS HERE ---
# We switch to Llama 3.1 Instruct (4-bit).
# "Instruct" means it is trained to chat and follow commands.
model_id = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"

# 1. Load Model & Tokenizer
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4", # Normalized float 4 (better accuracy)
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map="auto"
)

# Mandatory mapping for Llama 3.1
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:246: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [5]:
print(f"The loaded model is: {model_id}")

The loaded model is: unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit


In [6]:
# 2. Define Messages (No manual formatting needed!)
messages = [
    {"role": "system", "content": "You are a helpful AI assistant."},
    {"role": "user", "content": "Write a python script to verify an email address using regex."}
]

# 3. Apply Chat Template
# This works automatically now because the Instruct model has the config
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

# 4. Stream & Generate
streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

print(f"--- Loaded {model_id} ---\n")

with torch.no_grad():
    model.generate(
        **inputs, # Changed from `inputs` to `**inputs` to unpack input_ids and attention_mask
        max_new_tokens=1024,
        streamer=streamer,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        temperature=0.6,
        top_p=0.9
    )

--- Loaded unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit ---

**Email Verification using Regex in Python**

Below is a simple Python script that uses regular expressions to verify if a given email address is valid or not. This script checks for the basic structure of an email address, which includes a local part, an "@" symbol, a domain, and a top-level domain.

```python
import re

def verify_email(email):
    """
    Verify if the given email address is valid using regex.
    
    Args:
    email (str): The email address to be verified.
    
    Returns:
    bool: True if the email address is valid, False otherwise.
    """
    pattern = r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$"
    return bool(re.match(pattern, email))

# Example usage:
email_addresses = ["test@example.com", "invalid_email", "test@example"]

for email in email_addresses:
    if verify_email(email):
        print(f"{email} is a valid email address.")
    else:
        print(f"{email} is not a valid email addr

In [7]:
!pip install FastAPI

In [ ]:
import torch
import gc
import traceback  # <--- NEW: To see the real error
from flask import Flask, request, jsonify
from pyngrok import ngrok, conf
import logging
import os

# --- 1. CONFIG ---
LOG_FILE = "/content/colab_debug.log"
logging.basicConfig(filename=LOG_FILE, level=logging.INFO, filemode="w")

# Console handler to see errors in Colab output immediately
console = logging.StreamHandler()
console.setLevel(logging.INFO)
logging.getLogger("").addHandler(console)

app = Flask(__name__)
PORT = 5000

@app.route("/health")
def health():
    return jsonify({"status": "UP"})

@app.route("/generate", methods=["POST"])
def generate():
    try:
        # 1. Cleanup Memory
        torch.cuda.empty_cache()
        gc.collect()

        data = request.json or {}
        user_prompt = str(data.get("prompt", ""))

        logging.info(f"Received prompt: {user_prompt[:30]}...")

        # 2. Prepare Messages
        messages = [
            {"role": "system", "content": "You are a helpful AI assistant."},
            {"role": "user", "content": user_prompt}
        ]

        # 3. Tokenize
        # We capture the raw output first
        inputs_data = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt"
        ).to("cuda")

        # --- SAFETY CHECK: Handle Dictionary vs Tensor ---
        # If the tokenizer gave us a dictionary (BatchEncoding), extract the IDs.
        if hasattr(inputs_data, "input_ids"):
            input_tensor = inputs_data.input_ids
        elif isinstance(inputs_data, dict) and "input_ids" in inputs_data:
            input_tensor = inputs_data["input_ids"]
        else:
            # It's already a tensor (List of IDs)
            input_tensor = inputs_data

        # 4. Generate
        with torch.no_grad():
            outputs = model.generate(
                input_ids=input_tensor,  # Pass the CLEAN tensor
                max_new_tokens=512,
                temperature=0.7,
                top_p=0.9,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )

        # 5. Decode
        # Slice off the prompt part
        response_ids = outputs[0][input_tensor.shape[1]:]
        response_text = tokenizer.decode(response_ids, skip_special_tokens=True)

        return jsonify({"response": response_text})

    except Exception as e:
        error_trace = traceback.format_exc()
        logging.error(f"CRASH REPORT:\n{error_trace}")
        return jsonify({"error": str(e), "trace": error_trace}), 500

@app.route("/logs")
def logs():
    if not os.path.exists(LOG_FILE): return "No logs"
    with open(LOG_FILE) as f:
        return "<pre>" + "".join(f.readlines()[-50:]) + "</pre>"

# --- START ---
ngrok.kill()
public_url = ngrok.connect(PORT).public_url
print(f"\n🚀 API IS LIVE: {public_url}")
app.run(port=PORT, use_reloader=False)


🚀 API IS LIVE: https://kyler-arrhythmical-overearnestly.ngrok-free.dev
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
 * Running on http://127.0.0.1:5000
 * Running on http://127.0.0.1:5000
 * Running on http://127.0.0.1:5000
 * Running on http://127.0.0.1:5000
 * Running on http://127.0.0.1:5000
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
Press CTRL+C to quit
Press CTRL+C to quit
Press CTRL+C to quit
Press CTRL+C to quit
Press CTRL+C to quit
Press CTRL+C to quit
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
INFO:werkzeug:127.0.0.1 - - [15/Feb/2026 16:58:20] "POST /generate HTTP/1.1" 200 -
127.0.0.1 - - [15/Feb/2026 16:58:20] "POST /generate HTTP/1.1" 200 -
127.0.0.1 - - [15/Feb/2026 16:58:20] "POST /generate HTTP/1.1" 200 -
127.

In [ ]:
# =====================================
# Flask + pyngrok — Colab Compatible
# =====================================

from flask import Flask, request, jsonify
from pyngrok import ngrok
import logging, os, torch

# ---------------- Logging ----------------
LOG_FILE = "/content/colab_debug.log"

logging.basicConfig(
    filename=LOG_FILE,
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    filemode="w"
)

console = logging.StreamHandler()
console.setLevel(logging.INFO)
logging.getLogger("").addHandler(console)

# ---------------- App ----------------
app = Flask(__name__)
PORT = 5000

@app.route("/health")
def health():
    return {"status": "UP"}

@app.route("/generate", methods=["POST"])
def generate():
    try:
        data = request.json or {}
        prompt = str(data.get("prompt", ""))

        logging.info(f"Prompt length: {len(prompt)}")

        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        input_len = inputs.input_ids.shape[1]

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=min(1024, 4096 - input_len),
                temperature=0.7,
                top_p=0.95,
                top_k=50,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )

        gen_tokens = outputs[0][input_len:]
        text = tokenizer.decode(gen_tokens, skip_special_tokens=True)

        logging.info(f"Generated length: {len(text)}")

        return jsonify({"generated_text": text})

    except Exception as e:
        logging.error(str(e))
        return jsonify({"error": str(e)}), 500

@app.route("/logs")
def logs():
    if not os.path.exists(LOG_FILE):
        return "No logs found"

    with open(LOG_FILE) as f:
        lines = f.readlines()

    return "<pre>" + "".join(lines[-100:]) + "</pre>"

# ---------------- Tunnel ----------------

ngrok.kill()  # Clear stale tunnels

public_url = ngrok.connect(PORT).public_url

print(f"\nPUBLIC URL → {public_url}")
print(f"LOGS → {public_url}/logs")

# ---------------- Run Server ----------------
app.run(port=PORT, use_reloader=False)


In [ ]:
!ngrok http 5000